Analyse de la qualité des donnnées.
##### Objectif :  Evaluer la qualité des données du dataset de marketing
    

Importation des bibliothèques nécessaires et chargement du dataset

In [ ]:
# Vérification du chemin sur lequel Python cherche ses modules
import os
print(os.getcwd())

In [ ]:
#permet de remonter au chemin du notebook afin d'importer les modules du dossier src
import sys
sys.path.append(os.path.abspath(".."))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
#chargement de l'extension IPython autoreload, qui va surveiller les fichiers python importés et va les recharger automatiquement
%load_ext autoreload 
# La syntaxe ci-dessous permet de recharger tous les modules importés avant chaque ligne de code. Cela permettra d'éviter de recharger le kernel après modification des fichiers de nos modules
%autoreload 2
from  src import test_stat, nettoyage,visualisation


In [ ]:
pd.set_option('display.max_columns', None)   # afficher toutes les colonnes
pd.set_option('display.max_colwidth', None)  # afficher tout le contenu des cellules

In [ ]:

# Chargement du dataset
df_raw=pd.read_csv("../Data/Raw/marketing_data.csv") #importation du dataset brute sous forme de dataframe
df=df_raw.copy() #copie du dataframe

In [ ]:
#Vérification de la réussite du chargement
df.head()

In [ ]:
# Liste les colonnes 
df.columns

In [ ]:
#Renomination des colonnes du dataset
df.rename(columns={'Education':'Niveau_Education','Year_Birth' : 'Annee_naissance','Marital_Status' : 'Statut_Marital',' Income ' : 'Revenu', 'Kidhome' : 'Nb_Enfants', 'Teenhome' : 'Nb_Ado', 'Dt_Customer': 'Date_Inscription', 'Recency':'Nbjour_Dernier_Achat','MntWines': 'Montant_Vins', 'MntFruits':'Montant_Fruits', 'MntMeatProducts':'Montant_Viandes', 'MntFishProducts':'Montant_Poissons', 'MntSweetProducts':'Montant_Sucreries','MntGoldProds':'Montant_Or', 'NumDealsPurchases':'NbAchat_Remise','NumWebPurchases':'NbAchat_SiteWeb', 'NumCatalogPurchases':'NbAchat_Catalogue','NumStorePurchases':'NbAchat_Magasin','NumWebVisitsMonth':'NbVisit_SiteWebCeMois','AcceptedCmp3':'Campagne3','AcceptedCmp4':'Campagne4', 'AcceptedCmp5':'Campagne5', 'AcceptedCmp2':'Campagne2', 'AcceptedCmp1':'Campagne1', 'Response':'DerniereCampagne','Complain':'Plaintes','Country':'Pays'}, inplace=True)


In [ ]:

#Afficher les 5 premières lignes pour vérifier
print(df.head().to_string(index=False))

Analyse et identification des incohérences et des données éronnées 

In [ ]:
#présentation générale  des informations du dataset et du nombre de valeurs non manquantes par colonne
df.shape #utile pour connaitre le nombre de ligne et de colonnes du dataset
df.info()

In [ ]:
#Fouille des colonnes afin d'identifier des anomalies
for val in df.select_dtypes(exclude=["int","datetime","float"]):
    print(f"---------{val}--------")
    print(df[val].unique())
    print(df[val].value_counts())

In [ ]:
#Filtrage de la valeur YOLO 
df.iloc[df["Statut_Marital"]=="YOLO"]

In [ ]:
#Filtrage de la valeure Absurd
df.iloc[df["Statut_Marital"]=="Absurd"]

In [ ]:
# Description statistique des variables catégorielles(moyenne, Q1,Q2,Q3, min, max, nombre de valeurs, ...)
df.describe(include="object")

In [ ]:
# Description statistique des variables numériques (moyenne, Q1,Q2,Q3, min, max, nombre de valeurs, ...)
df.describe(include="int") #ou df.describe()

#df.describe(include="all") #description statistique de tous les types de données

In [ ]:
df.info()

In [ ]:
#Fusion respective des colonnes Montants et Nb_Achats
df["Montant_Total"]=(
    df["Montant_Vins"]
    +df["Montant_Poissons"]
    +df["Montant_Fruits"]
    +df["Montant_Viandes"]
    +df["Montant_Sucreries"]
    +df["Montant_Or"]
)

df["NbAchat_Total"]=(
    df["NbAchat_Catalogue"]
    +df["NbAchat_Remise"]
    +df["NbAchat_SiteWeb"]
    +df["NbAchat_Magasin"]
)



In [ ]:
#Vérification de la cohérence entre le nombre d'achat total et nombre de jour effectué depuis le dernier achat
#Vérification de la cohérence entre le nombre d'achat total et le montant total dépensé en achat
df[(df["NbAchat_Total"]==0)&(df["Nbjour_Dernier_Achat"]!=0)]

Observation :
###### Nous remarquons que Date_Inscription est de type object ce qui n'est pas correcte. Nous allons donc le transformer en type date.
###### Nous remarquons les colonnes portant sur les campagnes et réponses sont de type int. Nous allons donc les transformer en booléen
###### Nous manipulons mieux les âges que les années. Nous allons insérer la colonne Age. Il sera calculé en tenant compte de 2014 comme année de référence car  dans le dataset, les activités se sont déroulées de 2012 à 2014,aussi les colonnes de la date d'inscription et du nombre jour effectué après le dernier acaht, nous font comprendre que les clients sont toujours actifs en 2014
###### Dans la colonne Niveau_Education nous avons remarqué Graduate qui n'est pas un niveau d'éducation. Nous devons donc le remplacer par Graduation
###### Dans la colonne Statut_Marital, nous avons d'un côté YOLO et Abusurd qui ne sont pas des statuts matrimoniaux et seront supprimés car ils sont très peu représentés(4 au total) dans la variable Statut_Marital.
###### De l'autre Alone et Single seront fusionnés car ils ont le même sens
###### Aussi, Married et Together seront fusionnés. En effet, nous avons étudié la similarité de Married et Together tant statistiquement(mediane par groupe des Revenu et Montant_Total) que graphiquement(boxplot)

###### Nous avons remarqué des clients qui ont effectué aucun achat(NbAchat_Total=0) mais ont dépensé de l'argent pour acheté certains produits, ce qui est incohérent. Ausssi, leurS nombres de jours effectués depuis le dernier achat est non nul alors qu'ils n'ont rien acheté.


Traitement des incohérences et données erronées

In [ ]:
#Conversion de la colonne Date_Inscription en format date et suppression
df['Date_Enrollement']=pd.to_datetime(df['Date_Inscription'])
df.drop('Date_Inscription', axis='columns', inplace=True) #suppression de la colonne Date_inscription

In [ ]:
#Conversion de int en bool des colonnes campagnes, reponses, plaintes
campagne_type=["Campagne1","Campagne2","Campagne3","Campagne4","Campagne5","DerniereCampagne", "Plaintes"]
for val in campagne_type:
    df[val]=df[val].astype(bool)

In [ ]:
#Trouvons l'âge des clients. Nous allons le faire en tenant compte de 2014
# Car les activités se sont déroulés de 2012 à 2014
from datetime import datetime
df['Age']=2014-df['Annee_naissance']

In [ ]:
#Remplacement de Graduation par Graduate 
df["Niveau_Education"]=df["Niveau_Education"].replace({"Graduation":"Graduate"}, inplace=True) #inplace=True permet de commiter les modifications dans tout le dataset

In [ ]:
#Fusion de Single et Alone
df["Statut_Marital"]=df["Statut_Marital"].replace("Alone","Single")

#Fusion de Married et Together
df["Statut_Marital"]=df["Statut_Marital"].replace("Together","Married")

#Suppression des valeurs YOLO et Together
df=df[~df["Statut_Marital"].isin(["YOLO","Absurd"])]

In [ ]:
#Suppression de tous les clients qui n'ont pas été enregistré comme acheteur mais ont quand-même dépensé de l'argent pour les produits de la compagnie
df=df[~((df["NbAchat_Total"]==0)&(df["Nbjour_Dernier_Achat"]!=0))]

Analyse et identifaction des doublons

In [ ]:
#nombre de lignes exactement identiques
df.duplicated().sum()

Observation: Il n'y a aucun doublon

Analyse et indentification des valeurs manquantes

In [ ]:
#comptage des valeurs manquantes par colonne
df.isnull().sum()

In [ ]:
df[df["Revenu"].isna()]

###### Nous remarquons que ces clients qui ont des revenus manquants,ont des dépenses enregistrées concernant les fruits, poissons et autres. Les valeurs manquantes ne signifient pas ces clients n'ont pas de revenu

In [ ]:
#Nous devons avant tout trouver le type de valeurs manquantes(MCAR, MAR,MNAR) avant tout traitement
#Nous suspectons les revenus manquants de dépendre du niveau d'éducation,du statu matrimonial, du nombre d'enfants et de l'âge
#Procédons au test de contingence(d'indépendance) du khi-deux
#Evalution de la relation entre les valeurs manquantes et les colonnes suspectées

# Transformons l'âge en groupe d'age (catégorie)
df["Groupe_Age"]=pd.cut(df["Age"],bins=[18,25,35,45,60,100],
labels=["18-25","25-35","35-45","45-60","60+"])

#Fusionnons le nombres d'enfants et ado
df["Total_Enfants"]=df["Nb_Enfants"] + df["Nb_Ado"]

#Transformons Total_Enfants en catégorie
df["Groupe_TotalEnfants"]=pd.cut(
    df["Total_Enfants"],
    bins=[0,1,2,3],
    labels=["0","1","2-3"],
    include_lowest=True #pour inclure la valeur la plus basse de l'intervalle car pandas par défaut l'exclut
    )

In [ ]:
#Evaluation de la dépendance entre les valeurs manquantes et les variables suspectées
#Appel de la fonction test_contingence depuis le fichier test_stat.py
var=["Groupe_Age","Niveau_Education","Statut_Marital","Groupe_TotalEnfants"]
for i in var:
    test_stat.test_contingence(df,"Revenu",i)

Interprétation :
###### Le test de contingence indique qu'il existe  relation entre les revenus manquants et la variables Groupe_TotalEnfants. De ce fait, les variables manquantes  sont de type MAR.

Imputation et traitement des valeurs manquantes
###### Nous devons choisir entre l'imputation par la moyenne et celle par la médiane. Il faut donc étudier la normalité de la distribution de la série de données afin de faire le choix

In [ ]:
#Vérifions la nature de la distribution de la série de données(Revenu)
#Vérification de la normalisattion des données
# appel de la fonction test_shapiro depuis le fichier test_stat
test_stat.test_shapiro(df)


Observation:
###### Les données de la colonne Revenu ne suivent pas la loi normale. Nous optons donc pour l'imputation par la médiane. Nous allons faire une imputation conditionnelle. C'est-à-dire, en tenant compte des différents groupes des revenus des clients par Groupe_TotalEnfants nous allons remplacer chaque valeur manquante par la médiane de son groupe correspondant

In [ ]:
#Imputation conditionnelle
df["Revenu"]=df["Revenu"].fillna(df.groupby("Groupe_TotalEnfants")["Revenu"].transform("median"))

Conclusion : Les valeurs manquantes sont  traitées

Analyse et identification des valeurs aberrantes

###### Nous connaissons déjà que les donnéees ne suivent pas la loi normale. Dans ce cas, nous allons procéder par une analyse graphique et  une analyse statistique basée sur la méthode de l'IQR pour identifier et analyser les valeurs aberrantes. Si les données suivaient la loi normale, nous aurions procédé à une analyse statistique basée sur la méthode du Z-score

In [ ]:
#Analyse graphique : Cherchons les outliers
for col in ["Revenu","Montant_Total","Age", "NbAchat_Total", "Nbjour_Dernier_Achat"]:
    visualisation.graphe_histplot(df,col)

Rappel : 
###### Nous avons regroupé en amont les différents Montants et Nb_Achat respectivement en Montant_Total et NbAchat_Total afin d'éviter de considérer comme aberrant une préférence de produit ou de canal d'achat

In [ ]:
#Appel de la fonction valeur_aberrante du fichier nettoyage.py en considérant les variables numériques
for val in ["Revenu","Montant_Total","Age", "NbAchat_Total", "Nbjour_Dernier_Achat"]:
   nettoyage.valeur_aberrante(df,val)
   visualisation.graphe_histplot(df,val)

In [ ]:
for val in df[["Revenu","Montant_Total","NbAchat_Total"]]:
    print(f"---------{val}-----------")
    print(df.groupby("Statut_Marital")[val].median())

In [ ]:
#Inspection des revenus aberrants
df.iloc[df["Revenu"]>150000]

In [ ]:
#Inspection des montants totaux aberrants
df.iloc[df["Montant_Total"]>2000]

In [ ]:
#Inspection des nombre d'achat totaux aberrants
df.iloc[df["NbAchat_Total"]>40]

In [ ]:
#Inspection des âges aberrants
df.iloc[df["Age"]>100]

In [ ]:
#Inspection du revenu extrême
df.loc[df["Revenu"]==666666]

In [ ]:
#Vérification du coeffcicient d'asymétrie des colonnes Age, MOntant_Total, NbAchat_Total et Revenu
for val in ["Revenu","Montant_Total","Age", "NbAchat_Total"]:
   nettoyage.asymetrie(df,val)


In [91]:
#Nous optons pour la winorisation après avoir testé les transformations logarithmiques(les outliers étaient passés de 5 à 38 après 
# transformation et le taux d'asymétrie de 6,93 à -1,13) et racine carré(les outliers étaient passés de 5 à 12 arès transformation avec des outliers 
#extrêmes et l'asymétrie de 6,93 à 0,35)

for col in ["Revenu","Age"]:
    nettoyage.capping(df,col)


Revenu_capped
0       84835.00
1       57091.00
2       67267.00
3       32474.00
4       21474.00
          ...   
2235    66476.00
2236    31056.00
2237    46310.00
2238    65819.00
2239    94199.86
Name: Revenu_capped, Length: 2232, dtype: float64
Age_capped
0       44
1       53
2       56
3       47
4       25
        ..
2235    38
2236    37
2237    38
2238    36
2239    45
Name: Age_capped, Length: 2232, dtype: int64


In [ ]:
for col in ["Revenu","Age"]:
    nettoyage.valeur_aberrante(df,col)
    visualisation.graphe_boxplot

Conclusion:
###### Il sied de souligner que toutes les valeurs aberrantes sont pas des erreurs. C'est le cas des colonnes Montant_Total et NbAchat_Total qui présentent des outliers très proches de la borne supérieure. Elles peuvent être expliquer par le fait que certains clients ont un pouvoir d'achat élevé

###### L'analyse tant statistque que graphique de la colonne Revenu nous permis de constater des valeurs aberrantes assez proches de la borne supérieure et une très éloignéé(extrême) qui est 666666. Nous l'avons donc supprimé car  sa proportion est très faible. 
###### Aussi, d'autres valeurs aberrantes ont été acceptées car elles se rapprochent de la tendance des des revenus des clients

###### L'analyse tant statistique que graphique de la colonne Age nous permis de connstater trois valeurs aberrantes qui sont 114, 115 et 121ans. Ces valeurs sont loin d'être plausibles et tenant compte de leur faible proportion sur l'âge des clients nous les avons supprimé

Fin de la phase de nettoyage et export du dataset en fichier csv

In [ ]:
df.to_csv("C:/Users/Kardo BALOSSA/Data_Analyis_Projects/marketing-data-analyst-project/Data/Clean/dataset_cleaned.csv",index=False)